<a href="https://colab.research.google.com/github/Vykhrystenko/ML_FIT_3-9_Vykhrystenko/blob/main/%D0%9B%D0%A0_9_%D0%92%D0%B8%D1%85%D1%80%D0%B8%D1%81%D1%82%D0%B5%D0%BD%D0%BA%D0%BE_3_9_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Вихристенко В. О.

**Група:** 3-9  

**Дата:** 13.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [17]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [18]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [19]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [20]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print("Кількість унікальних користувачів:", ratings_df['user_id'].nunique())
print("Кількість унікальних фільмів:", ratings_df['item_id'].nunique())

print("Мінімальна оцінка:", ratings_df['rating'].min())
print("Максимальна оцінка:", ratings_df['rating'].max())
print("Середня оцінка:", round(ratings_df['rating'].mean(), 2))

print("Пропущені значення у датасеті:")
print(ratings_df.isnull().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53
Пропущені значення у датасеті:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64


**Короткі спостереження щодо даних:**

Набір даних містить 100 000 оцінок від 943 унікальних користувачів для 1682 фільмів. Оцінки варіюються від 1.0 до 5.0, при цьому середня оцінка становить близько 3.53. Пропущені значення у таблиці відсутні.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [21]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

In [22]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(zip(
    test_df['user_id'],
    test_df['item_id'],
    test_df['rating']
))

print("Діапазон шкали оцінок:", reader.rating_scale)
print("Кількість елементів у testset:", len(testset))
print("Навчальні дані успішно перетворено для Surprise!")

Діапазон шкали оцінок: (1, 5)
Кількість елементів у testset: 20000
Навчальні дані успішно перетворено для Surprise!


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [23]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._


Параметри моделей впливають на їхню точність та здатність до узагальнення. Для SVD кількість латентних факторів (n_factors) та швидкість навчання (lr_all) контролюють складність і швидкість збігання апроксимації, а регуляризація (reg_all) запобігає перенавчанню. Для KNNBasic кількість сусідів (k) та метрики подібності (косинусна і Пірсона) визначають силу зв'язків між уподобаннями користувачів чи фільмів, забезпечуючи баланс між локальним контекстом і загальними трендами.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [24]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [25]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [26]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print(" SVD найкращі результати ")
print("Найкращий RMSE:", round(grid_search_svd.best_score['rmse'], 4))
print("Найкращий MAE:", round(grid_search_svd.best_score['mae'], 4))
print("Найкращі параметри:", grid_search_svd.best_params['rmse'])

print("\n KNNBasic найкращі результати ")
print("Найкращий RMSE:", round(grid_search_knn.best_score['rmse'], 4))
print("Найкращий MAE:", round(grid_search_knn.best_score['mae'], 4))
print("Найкращі параметри:", grid_search_knn.best_params['rmse'])

 SVD найкращі результати 
Найкращий RMSE: 0.944
Найкращий MAE: 0.7442
Найкращі параметри: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

 KNNBasic найкращі результати 
Найкращий RMSE: 1.0285
Найкращий MAE: 0.8154
Найкращі параметри: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9416 | 0.7431 | {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} |
| KNNBasic | 1.0287 | 0.8160 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._


За результатами перехресної перевірки (крос-валідації) модель SVD продемонструвала значно вищу точність, ніж KNNBasic (менші значення як RMSE — 0.9416 проти 1.0287, так і MAE — 0.7431 проти 0.8160).

Найкращими параметрами для SVD стали: кількість латентних факторів n_factors = 50, швидкість навчання lr_all = 0.01 та коефіцієнт регуляризації reg_all = 0.05. Для алгоритму KNNBasic оптимальними виявилися: кількість сусідів k = 40, мінімальна кількість сусідів min_k = 1, а також використання косинусної міри схожості на основі користувачів (user_based: True). Менша похибка SVD пояснюється її здатністю ефективніше виявляти приховані взаємозв'язки між користувачами та фільмами у матриці оцінок порівняно з методом K-найближчих сусідів.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [27]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_svd.fit(trainset)

best_knn = KNNBasic(**best_knn_params)
best_knn.fit(trainset)

print("Остаточні моделі успішно навчені на повній навчальній вибірці!")
print("Параметри SVD:", best_svd_params)
print("Параметри KNNBasic:", best_knn_params)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Остаточні моделі успішно навчені на повній навчальній вибірці!
Параметри SVD: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Параметри KNNBasic: {'k': 40, 'min_k': 5, 'sim_options': {'name': 'cosine', 'user_based': True}}


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [28]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

In [29]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results_data = [
    {"Model": "SVD", "RMSE": round(svd_rmse, 4), "MAE": round(svd_mae, 4)},
    {"Model": "KNNBasic", "RMSE": round(knn_rmse, 4), "MAE": round(knn_mae, 4)}
]

results_df = pd.DataFrame(results_data).sort_values(by="RMSE")
print("\nТаблиця тестових результатів:")
display(results_df)


Таблиця тестових результатів:


,Model,RMSE,MAE
0,SVD,0.9192,0.7234
1,KNNBasic,1.0117,0.7993


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Оцінки на тестовій вибірці практично збігаються з результатами крос-валідації (для SVD тестовий RMSE — 0.9192, для KNNBasic — 1.0120). Що свідчить про відсутність перенавчання та стабільність моделей. Модель SVD стабільно демонструє вищу точність порівняно з KNNBasic.


## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [30]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
user_id = '196'

best_model = best_svd

try:
    inner_uid = trainset.to_inner_uid(user_id)
    print(f"Користувач {user_id} присутній у навчальній вибірці (внутрішній ID: {inner_uid}).")
except ValueError:
    print(f"Користувач {user_id} відсутній у навчальній вибірці.")

rated_items = set([iid for (iid, rating) in trainset.ur[trainset.to_inner_uid(user_id)]])

all_item_inner_ids = trainset.all_items()
all_raw_items = [trainset.to_raw_iid(inner_id) for inner_id in all_item_inner_ids]

unseen_items = [iid for iid in all_raw_items if iid not in rated_items]

predictions = []
for iid in unseen_items:
    pred = best_model.predict(uid=user_id, iid=iid)
    predictions.append((iid, pred.est))

predictions.sort(key=lambda x: x[1], reverse=True)
top_10 = predictions[:10]

top_10_df = pd.DataFrame(top_10, columns=["Item ID", "Estimated Rating"])
print(f"\nТоп-10 рекомендованих фільмів для користувача {user_id}:")
display(top_10_df)

Користувач 196 присутній у навчальній вибірці (внутрішній ID: 606).

Топ-10 рекомендованих фільмів для користувача 196:


,Item ID,Estimated Rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,285,4.391462
8,656,4.385379
9,22,4.377388


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

За результатами перехресної перевірки було обрано модель SVD, оскільки вона продемонструвала вищу точність прогнозування порівняно з KNNBasic завдяки ефективному виявленню латентних взаємозв'язків.

Найкращими гіперпараметрами для SVD стали n_factors = 50, lr_all = 0.01 та reg_all = 0.05, а для KNNBasic — k = 40, min_k = 1 з косинусною мірою схожості.

На незалежній тестовій вибірці модель SVD досягла найкращих показників із Test RMSE 0.9192 та Test MAE 0.7234. Водночас алгоритм KNNBasic показав дещо нижчу точність із Test RMSE 1.0120 та Test MAE 0.7984. Оцінки якості на тестовій вибірці практично збігаються з результатами крос-валідації, що свідчить про відсутність перенавчання та високу стабільність моделей.

Головним обмеженням побудованих систем є неможливість формування якісних рекомендацій для нових користувачів або нещодавно доданих фільмів, які ще не мають накопиченої історії оцінок.

Під час виконання даної лабораторної роботи було завантажено набір даних MovieLens 100K та побудовано дві рекомендаційні моделі. Для цільового користувача також було сформовано якісний персоналізований список із топ-10 найкращих рекомендацій фільмів.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**